# Module 2: What Exactly is an RDD?

Objective: By the end of this chapter, students should not only know the definition of an RDD but also understand why every word in RDD exists, how Spark uses it internally, and how it differs from normal collections.


numbers = [10, 20, 30, 40, 50]

Where is this data stored?

Inside your computer’s RAM.

Everything happens inside one machine.

RDD
↓
Resilient Distributed Dataset

What Does “Resilient” Mean?

Resilient means--> Able to recover from failure.

### Why is RDD Immutable?

Because immutability:

- Makes fault recovery easier.
- Avoids conflicts when many tasks run in parallel.
- Simplifies optimization and lineage tracking.

### Advantages of RDD

- Fault tolerant
- Distributed processing
- Parallel execution
- Scalable to many machines
- Supports lazy evaluation
- Works well for low-level transformations and custom processing

### Limitations of RDD

RDDs also have drawbacks.

- No automatic query optimization.
- No schema information.
- More verbose code.
- DataFrames and Datasets are generally preferred for structured data because Spark can optimize them better.


### Full Internal Flow

Python Program
↓
SparkSession
↓
SparkContext
↓
Driver
↓
RDD Created
↓
Partitions Planned
↓
(No execution yet)
↓
collect()
↓
Job
↓
Stage
↓
Task
↓
Executor
↓
Result
↓
Driver


In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("RDD Fundamentals")
    .master("local[2]")
    .getOrCreate()
)

sc = spark.sparkContext

print(sc.appName)
print(sc.uiWebUrl)

In [ ]:
numbers=[10,20,30,40,50]

In [ ]:
rdd=sc.parallelize(numbers)
print("RDD is Created")

In [ ]:
rdd.collect()

In [ ]:
spark.stop()

# Creating RDD 

In [ ]:
spark.stop()

In [ ]:
from pyspark.sql import SparkSession
spark=(
    SparkSession.builder
    .appName("RDD Fundamentals")
    .master("local[*]")
    .getOrCreate()
)
sc=spark.sparkContext
print(sc.uiWebUrl)

sc

### Method 1

sc.parallelize() - 

In [ ]:
numbers=[1,2,3,4,5,6,7,8,9,10]

print(type(numbers))

In [ ]:
rdd = sc.parallelize(numbers,3)

In [ ]:
print(type(rdd))

In [ ]:
rdd.getNumPartitions()

In [ ]:
rdd.glom().collect()

In [ ]:
rdd.collect()

In [ ]:
rdd1=sc.parallelize(range(1,11),5)
print(rdd1.glom().collect())

### Method 2 : sc.textFile()



In [ ]:
employee_rdd=sc.textFile("employee.txt")

In [ ]:
employee_rdd.collect()

In [ ]:
employee_rdd.getNumPartitions()

In [ ]:
employee_rdd.glom().collect()

# RDD Partitions 

### What is Partition?

- A Logical chucnk of an RDD that can be processed independently 

- RDD =[1,2,3,4,5,6,7,8] - 2 PARTITIONS 

RDD 
|_ PARTITION 0 [1,2,3]
|
|- PARTITION 1 [8,8,9]

### Why Does Spark needs Partitions 

- To distribute the data across multiple nodes in a cluster for parallel processing.




In [ ]:
numbers = list(range(1,21))

In [ ]:
numbers

In [ ]:
rdd=sc.parallelize(numbers,4)

In [ ]:
# Check partition count 

rdd.getNumPartitions()

In [ ]:
# Actual Contents inside partition 

rdd.glom().collect()

# What does Glom()() do?    
# Glom() converts each partition into a list and returns an RDD of lists.



# Maximum parallel tasks at one time ---> Total avilable cores in the cluster. 

### Too Few Partition 
- Underutilization of resources.

### Too Many Partitions 
- Overhead of managing too many small tasks.

- Problems
- Task scheduling overhead 
- too much metadata 
- Excessive task launch cost 
- Possible many tiny output files 
- Driver schdeduling pressre 

### What Determines Partition Count 

- Number of cores in the cluster

- For sc.textFile() partition size depends on:

- input files 
- file sizes 
- Hadppod input slpits 
- clock/split settings 
- compression 
- filesystem 
- Spark Configuration 






In [ ]:
spark.stop()

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("TooManyPartitions")
    .master("local[*]")
    .getOrCreate()
)

sc = spark.sparkContext

print("Spark Version:", sc.version)
print(sc.appName)
print(sc.uiWebUrl)
print("Available Cores:", sc.defaultParallelism)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/14 20:55:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Version: 3.5.7
TooManyPartitions
http://macbookair.lan:4040
Available Cores: 8


In [ ]:
rdd_normal=sc.parallelize(range(1_000_000),8)
print("Number of Partitions:", rdd_normal.getNumPartitions())

In [ ]:
import time 
start_time=time.time()
result=rdd_normal.map(lambda x:x*2).sum()
end_time=time.time()
print("Result:", result)
print("Time taken:", end_time - start_time)

In [ ]:
rdd_many=sc.parallelize(range(1_000_000),10000)
print("Number of Partitions:", rdd_many.getNumPartitions())

In [ ]:
import time 
start_time=time.time()
result=rdd_many.map(lambda x:x*2).sum()
end_time=time.time()
print("Result:", result)
print("Time taken:", end_time - start_time)

In [ ]:
import time
partition_count = [
    2,
    4,
    8,
    16,
    100,
    1000,
    5000
]

for partitions in partition_count:
    rdd=sc.parallelize(range(1_000_000),partitions)
    start_time=time.time()
    result=(
        rdd.map(lambda x:x*2).filter(lambda x:x%3==0).sum()
    )
    end_time=time.time()
    print(
        f"Partitions:{partitions:<5}"
        f"Time: {end_time-start_time:.4f} seconds"
    )

In [ ]:
rdd=sc.parallelize(range(1,13),4)

In [ ]:
def show_partitions(index,iterator):
    for value in iterator:
        yield(index,value)
result=rdd.mapPartitionsWithIndex(show_partitions)
result.collect()


# RDD Lazy Evalaution and Lineage 

- Transformations such as map() and filter() do not execute immediately. Spark remembers them and waits until an action is called.

### What is Lazy Evaluation

- Spark delays execution of transformations until it actually needs a result.

- When you write them, Spark builds lineage/dependencies.

- It doesn’t immediately run the entire computation.


rdd = sc.parallelize([1, 2, 3, 4, 5])

result = rdd.map(lambda x: print("Processing:", x))

Output :

Processing: 1
Processing: 2
Processing: 3
Processing: 4
Processing: 5

- But simply defining the transformation does not require Spark to process all records.

- result.collect()

Now the task actually executes.

- In local mode, you’ll see task-side output depending on the notebook/log environment.

map()
   ↓
No action
   ↓
No full Spark job

collect()
   ↓
Action
   ↓
Job starts

# Transformation vs Action

### Transformation

- Takes one RDD and produces another RDD.

RDD1
 ↓
map
 ↓
RDD2

- Transformations are normally lazy. 

### Action

- Produces a final result or writes output.

RDD
 ↓
Action
 ↓
Spark Job









RDD transformations create a **new RDD from an existing RDD**.

> **Important:** Transformations are **lazy**. Spark does not execute them immediately. Execution starts when an **Action** is called.

---

## Complete RDD Transformation List

```text
RDD TRANSFORMATIONS
│
├── 1. BASIC / ELEMENT TRANSFORMATIONS
│   │
│   ├── map()
│   ├── flatMap()
│   ├── filter()
│   ├── mapPartitions()
│   ├── mapPartitionsWithIndex()
│   ├── glom()
│   ├── sample()
│   ├── pipe()
│   └── keyBy()
│
├── 2. SET / DATASET TRANSFORMATIONS
│   │
│   ├── union()
│   ├── distinct()
│   ├── intersection()
│   ├── subtract()
│   └── cartesian()
│
├── 3. PAIR RDD — AGGREGATION TRANSFORMATIONS
│   │
│   ├── reduceByKey()
│   ├── groupByKey()
│   ├── aggregateByKey()
│   ├── combineByKey()
│   ├── foldByKey()
│   └── groupBy()
│
├── 4. PAIR RDD — VALUE TRANSFORMATIONS
│   │
│   ├── mapValues()
│   ├── flatMapValues()
│   ├── keys()
│   └── values()
│
├── 5. SORTING TRANSFORMATIONS
│   │
│   ├── sortByKey()
│   └── sortBy()
│
├── 6. JOIN / COGROUP TRANSFORMATIONS
│   │
│   ├── join()
│   ├── leftOuterJoin()
│   ├── rightOuterJoin()
│   ├── fullOuterJoin()
│   ├── cogroup()
│   └── groupWith()
│
├── 7. PARTITION TRANSFORMATIONS
│   │
│   ├── partitionBy()
│   ├── repartition()
│   └── coalesce()
│
└── 8. OTHER / SPECIAL TRANSFORMATIONS
    │
    ├── zip()
    ├── zipWithIndex()
    ├── zipWithUniqueId()
    ├── randomSplit()
    └── repartitionAndSortWithinPartitions()

# RDD Transformations — Narrow vs Wide

RDD transformations are mainly classified into **Narrow Transformations** and **Wide Transformations** based on the dependency between partitions.

---

## Narrow Transformations vs  Wide Transformations

```text
RDD TRANSFORMATIONS
│
├── NARROW TRANSFORMATIONS
│   │
│   │   Child partition depends on limited parent partition(s)
│   │   Normally NO SHUFFLE
│   │
│   ├── BASIC / ELEMENT
│   │   ├── map()
│   │   ├── flatMap()
│   │   ├── filter()
│   │   ├── mapPartitions()
│   │   ├── mapPartitionsWithIndex()
│   │   ├── glom()
│   │   ├── sample()
│   │   ├── pipe()
│   │   └── keyBy()
│   │
│   ├── SET / DATASET
│   │   ├── union()
│   │   └── cartesian()                  ← Special case
│   │
│   ├── PAIR RDD / VALUE
│   │   ├── mapValues()
│   │   ├── flatMapValues()
│   │   ├── keys()
│   │   └── values()
│   │
│   ├── PARTITION
│   │   └── coalesce()                   ← shuffle=False
│   │
│   └── OTHER / SPECIAL
│       ├── zip()
│       ├── zipWithIndex()               ← Special case
│       ├── zipWithUniqueId()
│       └── randomSplit()
│
│
└── WIDE TRANSFORMATIONS
    │
    │   Data generally needs redistribution across partitions
    │   SHUFFLE is normally required
    │
    ├── SET / DATASET
    │   ├── distinct()
    │   ├── intersection()
    │   └── subtract()
    │
    ├── AGGREGATION
    │   ├── groupBy()
    │   ├── reduceByKey()
    │   ├── groupByKey()
    │   ├── aggregateByKey()
    │   ├── combineByKey()
    │   └── foldByKey()
    │
    ├── SORTING
    │   ├── sortByKey()
    │   └── sortBy()
    │
    ├── JOIN / COGROUP
    │   ├── join()
    │   ├── leftOuterJoin()
    │   ├── rightOuterJoin()
    │   ├── fullOuterJoin()
    │   ├── cogroup()
    │   └── groupWith()
    │
    ├── PARTITION
    │   ├── partitionBy()
    │   ├── repartition()
    │   └── coalesce(shuffle=True)
    │
    └── OTHER
        └── repartitionAndSortWithinPartitions()
```

---

## Rule

### Narrow Transformation

**One child partition depends on a limited number of parent partitions.**

```text
Parent RDD                 Child RDD

Partition 1  ────────────► Partition 1

Partition 2  ────────────► Partition 2

Partition 3  ────────────► Partition 3

                No Shuffle
```

Examples:

- `map()`
- `flatMap()`
- `filter()`
- `mapPartitions()`
- `mapValues()`

---

### Wide Transformation

**A child partition may require data from multiple parent partitions.**

```text
Parent RDD                      Child RDD

Partition 1 ─────┬────────────► Partition 1
                 │
Partition 2 ─────┼────────────► Partition 2
                 │
Partition 3 ─────┴────────────► Partition 3

                  SHUFFLE
```

Examples:

- `reduceByKey()`
- `groupByKey()`
- `distinct()`
- `sortByKey()`
- `repartition()`
- `join()`

---

## Key Difference

| Narrow Transformation         | Wide Transformation |
|---|---|
| Normally no shuffle           | Normally causes shuffle |
| Data stays locally accessible | Data may move across partitions/executors |
| Faster | More expensive       |
| Usually stays in same stage   | Creates a stage boundary |
| Less network I/O              | More network I/O |
| Example: `map()`              | Example: `reduceByKey()` |

---

## Remember

```text
NARROW
   │
   ├── No Shuffle
   │
   ├── Less Network I/O
   │
   ├── Faster
   │
   └── Same Stage
           

WIDE
   │
   ├── Shuffle
   │
   ├── Network I/O
   │
   ├── More Expensive
   │
   └── New Stage Boundary
```

> **Golden Rule:**  
> **Narrow → No Shuffle → Same Stage**  
> **Wide → Shuffle → Stage Boundary**

In [ ]:
numbers=[1,2,3,4,5]
result=[x *2 for x in numbers]
print(result)

In [ ]:
rdd=sc.parallelize([1,2,3,4,5])
result=rdd.map(lambda x: print("Processing",x))

In [ ]:
result.collect()

## Transformation vs Action 

- Transformation: Takes one RDD and produces another RDD 

- rdd2=rdd1.map(lambda x:x*2)

- Transformations are lazy 

## Action 

- Produces a final result or write a output 
- rdd.collect(), rdd.count(), rdd.first(), rdd.take(),rdd.reduce()

In [ ]:
rdd1=sc.parallelize(range(1,11))
rdd2=rdd1.map(lambda x:x*2)
rdd3=rdd2.filter(lambda x:x>10)
rdd4=rdd3.map(lambda x:(x,x**2))

In [ ]:
rdd4.collect()

In [ ]:
rdd1=sc.parallelize(range(1,11),2)
rdd2=rdd1.map(lambda x:x*2)
rdd3=rdd2.filter(lambda x:x>10)

In [ ]:
print(rdd3.toDebugString())

b'(2) PythonRDD[5] at RDD at PythonRDD.scala:53 []\n |  
ParallelCollectionRDD[4] at readRDDFromFile at PythonRDD.scala:289 []'

In [ ]:
print("rdd1")
print(rdd1.toDebugString())

print("\nrdd2")
print(rdd2.toDebugString())

print("\nrdd3")
print(rdd3.toDebugString())

In [ ]:
print(rdd1.glom().collect())

In [ ]:
print(rdd2.glom().collect())

In [ ]:
print(rdd3.glom().collect())

# Narrow and Wide Transformation 



In [ ]:
rdd1=sc.parallelize([1,2,3,4,5,6,7,8],2) # Parent RDD 
rdd2=rdd1.map(lambda x:x*2) # Child RDD 

# For Creating one partiton of RDD2, How many partition of RDD1 are needed 

## Narrow Transformation 

- Each child partition depends on only a small/fixed number of parent partition 

- No Shullfle 


In [ ]:
# Practice -1 

orders = [
    (
        "O101",
        "C101",
        "india",
        [
            ("Laptop", 50000, 1),
            ("Mouse", 1000, 2)
        ]
    ),
    (
        "O102",
        "C102",
        "usa",
        [
            ("Keyboard", 2000, 1),
            ("Monitor", 15000, 2)
        ]
    ),
    (
        "O103",
        "C103",
        "india",
        [
            ("Mobile", 30000, 1),
            ("Charger", 1500, 2),
            ("Cover", 500, 1)
        ]
    ),
    (
        "O104",
        "C104",
        "uk",
        [
            ("Tablet", 25000, 1)
        ]
    )
]

rdd = sc.parallelize(orders, 2)

# Requirments:

# Map() - clean the order 
# TO convert the country to uppercase

# flatmap() : Explode products 

# Calculate product total = Product_total=unit_price*quantity 

    # Return 

        # (
        #         order_id,
        #         c_id,
        #         country,
        #         product,
        #         unit_price,
        #         quantity,
        #         product_total
        # )

# Filter: Keep only product records where product_total >=20000









(
    order_id,
    cutomer_id,
    conuntry,
    [
        (product,price,quantity),
        (product,price,quantity),
        (product,price,quantity)
    ]
)

## mapPartitions()
 - function receives one entire partition as an iterator.

## mapPartitionsWithIndex()

In [2]:
rdd=sc.parallelize(
    [10,20,30,40,50.60],3
)

## What is Yield :




In [3]:
def numbers():
    return 10
    return 20
    return 30


print(numbers())

10


In [4]:
def numbers():
    yield 10
    yield 20
    yield 30

result=numbers()
print(result)

<generator object numbers at 0x10acbf530>


In [7]:
print(next(result))

30


## mapPartitionsWithIndex()

rdd.mappartitionswithIndex(functions)

- Partition index + Partition Iterator 



In [ ]:
rdd=sc.parallelize(
    [10,20,30,40,50,60]
)

### Practice Questions :

transections=[
    ("T002","C101",5000),
    ("T003","C102",35000),
    ("T004","C104",57000),
    ("T005","C105",55000),
    ("T006","C106",15000),
    ("T007","C107",70000),
    ("T008","C108",3000),
    ("T009","C109",45000),
    ("T00","C100",95000)
]




In [29]:
transection=[
    ("T002","C101",5000),
    ("T003","C102",35000),
    ("T004","C104",57000),
    ("T005","C105",55000),
    ("T006","C106",15000),
    ("T007","C107",70000),
    ("T008","C108",3000),
    ("T009","C109",45000),
    ("T00","C100",95000)
]

rdd1=sc.parallelize(transection,3)

In [26]:
def create_risk_client():
    print("Creating Risk API Client")
    return "RISK_CLIENT"

def get_risk(client,amount):
    if amount>=50000:
        return "HIGH"
    elif amount >=10000:
        return "medium"
    else:
        return "LOW"

    

In [27]:
def process_partition(records):
    client=create_risk_client()

    for transection in records:
        transection_id=transection[0]
        cutomer_id=transection[1]
        amount=transection[2]

        risk=get_risk(client,amount)

        yield(
            transection_id,
            cutomer_id,
            amount,
            risk
        )

In [ ]:
result=rdd.mapPartitions(process_partition)
result.collect()

Creating Risk API Client
Creating Risk API Client
Creating Risk API Client


[('T002', 'C101', 5000, 'LOW'),
 ('T003', 'C102', 35000, 'medium'),
 ('T004', 'C104', 57000, 'HIGH'),
 ('T005', 'C105', 55000, 'HIGH'),
 ('T006', 'C106', 15000, 'medium'),
 ('T007', 'C107', 70000, 'HIGH'),
 ('T008', 'C108', 3000, 'LOW'),
 ('T009', 'C109', 45000, 'medium'),
 ('T00', 'C100', 95000, 'HIGH')]

# ============================================================
# RDD PRACTICE QUESTIONS
# map() | flatMap() | filter() | mapPartitions()
# mapPartitionsWithIndex()
# ============================================================


# ============================================================
# 1. map() — Employee Salary & Bonus Calculation
# ============================================================

employees = [
    ("E101", "Anuj", "Data Engineer", 80000, 4.5),
    ("E102", "Rahul", "Developer", 60000, 3.8),
    ("E103", "Priya", "Data Engineer", 90000, 4.8),
    ("E104", "Amit", "Tester", 50000, 3.2),
    ("E105", "Neha", "Developer", 70000, 4.2)
]

rdd = sc.parallelize(employees, 3)


REQUIREMENT:

Using map(), calculate the annual bonus.

Rules:

rating >= 4.5
    → bonus = 20% of salary

rating >= 4.0 and < 4.5
    → bonus = 15% of salary

rating >= 3.5 and < 4.0
    → bonus = 10% of salary

rating < 3.5
    → bonus = 5% of salary


Return:

(
    employee_id,
    employee_name,
    role,
    salary,
    rating,
    bonus,
    final_salary
)


Where:

final_salary = salary + bonus


EXPECTED OUTPUT:

[
    ("E101", "Anuj", "Data Engineer", 80000, 4.5, 16000, 96000),

    ("E102", "Rahul", "Developer", 60000, 3.8, 6000, 66000),

    ("E103", "Priya", "Data Engineer", 90000, 4.8, 18000, 108000),

    ("E104", "Amit", "Tester", 50000, 3.2, 2500, 52500),

    ("E105", "Neha", "Developer", 70000, 4.2, 10500, 80500)
]



# ============================================================
# 2. map() — Transaction Risk Classification
# ============================================================

transactions = [
    ("T001", "C101", 5000, "INDIA"),
    ("T002", "C102", 25000, "USA"),
    ("T003", "C103", 75000, "INDIA"),
    ("T004", "C104", 120000, "UK"),
    ("T005", "C105", 8000, "INDIA")
]

rdd = sc.parallelize(transactions, 3)


REQUIREMENT:

Using map(), add a risk category.

Rules:

amount >= 100000
    → CRITICAL

amount >= 50000 and < 100000
    → HIGH

amount >= 10000 and < 50000
    → MEDIUM

amount < 10000
    → LOW


Return:

(
    transaction_id,
    customer_id,
    amount,
    country,
    risk
)


EXPECTED OUTPUT:

[
    ("T001", "C101", 5000, "INDIA", "LOW"),

    ("T002", "C102", 25000, "USA", "MEDIUM"),

    ("T003", "C103", 75000, "INDIA", "HIGH"),

    ("T004", "C104", 120000, "UK", "CRITICAL"),

    ("T005", "C105", 8000, "INDIA", "LOW")
]



# ============================================================
# 3. flatMap() — Order Product Explosion
# ============================================================

orders = [
    (
        "O101",
        "C101",
        [
            ("Laptop", 50000, 1),
            ("Mouse", 1000, 2)
        ]
    ),

    (
        "O102",
        "C102",
        [
            ("Keyboard", 2000, 1),
            ("Monitor", 15000, 2)
        ]
    ),

    (
        "O103",
        "C103",
        [
            ("Mobile", 30000, 1),
            ("Charger", 1500, 2),
            ("Cover", 500, 1)
        ]
    ),

    (
        "O104",
        "C104",
        []
    )
]

rdd = sc.parallelize(orders, 2)


REQUIREMENT:

Using flatMap(), convert each product inside an order
into an individual RDD record.

Return:

(
    order_id,
    customer_id,
    product_name,
    unit_price,
    quantity
)


EXPECTED OUTPUT:

[
    ("O101", "C101", "Laptop", 50000, 1),

    ("O101", "C101", "Mouse", 1000, 2),

    ("O102", "C102", "Keyboard", 2000, 1),

    ("O102", "C102", "Monitor", 15000, 2),

    ("O103", "C103", "Mobile", 30000, 1),

    ("O103", "C103", "Charger", 1500, 2),

    ("O103", "C103", "Cover", 500, 1)
]


Important:

O104 contains an empty product list.

So:

O104
    → 0 output records



# ============================================================
# 4. flatMap() — Application Log Word Extraction
# ============================================================

logs = [
    "ERROR database connection timeout",
    "INFO application started successfully",
    "WARN memory utilization high",
    "ERROR payment service unavailable"
]

rdd = sc.parallelize(logs, 2)


REQUIREMENT:

Using flatMap():

1. Split every log line into individual words.
2. Convert every word to uppercase.
3. Return every word as an individual RDD element.


EXPECTED OUTPUT:

[
    "ERROR",
    "DATABASE",
    "CONNECTION",
    "TIMEOUT",

    "INFO",
    "APPLICATION",
    "STARTED",
    "SUCCESSFULLY",

    "WARN",
    "MEMORY",
    "UTILIZATION",
    "HIGH",

    "ERROR",
    "PAYMENT",
    "SERVICE",
    "UNAVAILABLE"
]



# ============================================================
# 5. filter() — Fraud Transaction Filtering
# ============================================================

transactions = [
    ("T001", "C101", 5000, "INDIA"),
    ("T002", "C102", 60000, "INDIA"),
    ("T003", "C103", 120000, "USA"),
    ("T004", "C104", 8000, "UK"),
    ("T005", "C105", 90000, "INDIA"),
    ("T006", "C106", 150000, "SINGAPORE")
]

rdd = sc.parallelize(transactions, 3)


REQUIREMENT:

Using filter(), keep only suspicious transactions.

A transaction is suspicious when:

amount >= 100000

OR

country != "INDIA" AND amount >= 50000


EXPECTED OUTPUT:

[
    ("T003", "C103", 120000, "USA"),

    ("T006", "C106", 150000, "SINGAPORE")
]



# ============================================================
# 6. filter() — Data Quality Validation
# ============================================================

customers = [
    ("C101", "Anuj", 32, "anuj@gmail.com"),
    ("C102", "", 28, "rahul@gmail.com"),
    ("C103", "Priya", -5, "priya@gmail.com"),
    ("C104", "Amit", 45, ""),
    ("C105", "Neha", 25, "neha@gmail.com"),
    ("C106", "", 0, "")
]

rdd = sc.parallelize(customers, 3)


REQUIREMENT:

Using filter(), keep only valid customer records.

A valid record must satisfy all conditions:

name != ""

age > 0

email != ""


EXPECTED OUTPUT:

[
    ("C101", "Anuj", 32, "anuj@gmail.com"),

    ("C105", "Neha", 25, "neha@gmail.com")
]



# ============================================================
# 7. mapPartitions() — External API Enrichment
# ============================================================

transactions = [
    ("T001", "C101", 5000),
    ("T002", "C102", 15000),
    ("T003", "C103", 70000),
    ("T004", "C104", 3000),
    ("T005", "C105", 45000),
    ("T006", "C106", 90000)
]

rdd = sc.parallelize(transactions, 3)


Provided functions:

def create_risk_client():
    print("Creating Risk API Client")
    return "RISK_CLIENT"


def get_risk(client, amount):

    if amount >= 50000:
        return "HIGH"

    elif amount >= 10000:
        return "MEDIUM"

    else:
        return "LOW"



REQUIREMENT:

Using mapPartitions():

1. Create the Risk API client only once per partition.

2. Process all transactions inside that partition using
   the same client.

3. Calculate risk using get_risk().

4. Return:

(
    transaction_id,
    customer_id,
    amount,
    risk
)


EXPECTED OUTPUT:

[
    ("T001", "C101", 5000, "LOW"),

    ("T002", "C102", 15000, "MEDIUM"),

    ("T003", "C103", 70000, "HIGH"),

    ("T004", "C104", 3000, "LOW"),

    ("T005", "C105", 45000, "MEDIUM"),

    ("T006", "C106", 90000, "HIGH")
]


IMPORTANT:

Because there are 3 partitions:

create_risk_client()

should normally execute once per partition task attempt,
not once per transaction.



# ============================================================
# 8. mapPartitions() — Partition Level Data Quality Summary
# ============================================================

customers = [
    ("C101", "Anuj", 32, "anuj@gmail.com"),
    ("C102", "", 28, "rahul@gmail.com"),
    ("C103", "Priya", -5, "priya@gmail.com"),
    ("C104", "Amit", 45, ""),
    ("C105", "Neha", 25, "neha@gmail.com"),
    ("C106", "", 0, ""),
    ("C107", "Ravi", 40, "ravi@gmail.com"),
    ("C108", "Simran", 29, "simran@gmail.com")
]

rdd = sc.parallelize(customers, 4)


VALID RECORD RULE:

name != ""

age > 0

email != ""


REQUIREMENT:

Using mapPartitions():

Calculate one summary record for each partition.

Return:

(
    total_records,
    valid_records,
    invalid_records
)


For example:

If one partition contains:

C101 → VALID
C102 → INVALID

Then output for that partition should be:

(2, 1, 1)


EXPECTED OUTPUT FORMAT:

[
    (total, valid, invalid),
    (total, valid, invalid),
    (total, valid, invalid),
    (total, valid, invalid)
]


IMPORTANT:

Exact numbers depend on how Spark distributes the records.

There must be:

4 input partitions
        ↓
4 partition summary records



# ============================================================
# 9. mapPartitionsWithIndex() — Partition Distribution Analysis
# ============================================================

transactions = [
    ("T001", 5000),
    ("T002", 7000),
    ("T003", 9000),
    ("T004", 12000),
    ("T005", 15000),
    ("T006", 20000),
    ("T007", 25000),
    ("T008", 30000),
    ("T009", 35000),
    ("T010", 40000)
]

rdd = sc.parallelize(transactions, 4)


REQUIREMENT:

Using mapPartitionsWithIndex():

For every partition calculate:

1. Partition ID
2. Number of records
3. Total transaction amount


Return:

(
    partition_id,
    record_count,
    total_amount
)


EXPECTED OUTPUT FORMAT:

[
    (0, record_count, total_amount),

    (1, record_count, total_amount),

    (2, record_count, total_amount),

    (3, record_count, total_amount)
]


IMPORTANT:

Do not hard-code which transactions belong to which partition.

Use the iterator Spark passes to each partition.

There should be exactly:

4 partition summary records.



# ============================================================
# 10. mapPartitionsWithIndex() — ETL Audit Report
# ============================================================

records = [
    ("file1.csv", "T001", 5000, "SUCCESS"),
    ("file1.csv", "T002", 7000, "SUCCESS"),
    ("file1.csv", "T003", 0, "FAILED"),

    ("file2.csv", "T004", 12000, "SUCCESS"),
    ("file2.csv", "T005", 15000, "FAILED"),

    ("file3.csv", "T006", 20000, "SUCCESS"),
    ("file3.csv", "T007", 25000, "SUCCESS"),
    ("file3.csv", "T008", 30000, "SUCCESS"),

    ("file4.csv", "T009", 35000, "FAILED"),
    ("file4.csv", "T010", 40000, "SUCCESS")
]

rdd = sc.parallelize(records, 3)


REQUIREMENT:

Using mapPartitionsWithIndex():

Create one audit record per partition.

Calculate:

1. Partition ID
2. Total number of records
3. Number of SUCCESS records
4. Number of FAILED records
5. Total amount of SUCCESS records only


Return:

(
    partition_id,
    total_records,
    success_records,
    failed_records,
    total_success_amount
)


Example:

If Partition 0 contains:

("file1.csv", "T001", 5000, "SUCCESS")
("file1.csv", "T002", 7000, "SUCCESS")
("file1.csv", "T003", 0, "FAILED")


Then expected summary for that partition:

(
    0,
    3,
    2,
    1,
    12000
)


EXPECTED OUTPUT FORMAT:

[
    (
        0,
        total_records,
        success_records,
        failed_records,
        total_success_amount
    ),

    (
        1,
        total_records,
        success_records,
        failed_records,
        total_success_amount
    ),

    (
        2,
        total_records,
        success_records,
        failed_records,
        total_success_amount
    )
]


IMPORTANT:

Exact values depend on the actual records Spark places
inside each partition.



# ============================================================
# 11. COMBINED QUESTION
# map() + flatMap() + filter()
# ============================================================

orders = [
    (
        "O101",
        "C101",
        "india",
        [
            ("Laptop", 50000, 1),
            ("Mouse", 1000, 2)
        ]
    ),

    (
        "O102",
        "C102",
        "usa",
        [
            ("Keyboard", 2000, 1),
            ("Monitor", 15000, 2)
        ]
    ),

    (
        "O103",
        "C103",
        "india",
        [
            ("Mobile", 30000, 1),
            ("Charger", 1500, 2),
            ("Cover", 500, 1)
        ]
    ),

    (
        "O104",
        "C104",
        "uk",
        [
            ("Tablet", 25000, 1)
        ]
    )
]

rdd = sc.parallelize(orders, 2)


REQUIREMENT:


STEP 1 — map()

Convert country to uppercase.

Example:

"india"
    ↓
"INDIA"



STEP 2 — flatMap()

Convert every product into an individual record.

Return:

(
    order_id,
    customer_id,
    country,
    product,
    unit_price,
    quantity
)



STEP 3 — map()

Calculate:

product_total = unit_price * quantity


Return:

(
    order_id,
    customer_id,
    country,
    product,
    unit_price,
    quantity,
    product_total
)



STEP 4 — filter()

Keep only records where:

product_total >= 20000



EXPECTED FINAL OUTPUT:

[
    (
        "O101",
        "C101",
        "INDIA",
        "Laptop",
        50000,
        1,
        50000
    ),

    (
        "O102",
        "C102",
        "USA",
        "Monitor",
        15000,
        2,
        30000
    ),

    (
        "O103",
        "C103",
        "INDIA",
        "Mobile",
        30000,
        1,
        30000
    ),

    (
        "O104",
        "C104",
        "UK",
        "Tablet",
        25000,
        1,
        25000
    )
]



# ============================================================
# 12. COMBINED HARD QUESTION
# filter() + flatMap() + map() + mapPartitionsWithIndex()
# ============================================================

customers = [
    (
        "C101",
        "ACTIVE",
        [
            ("T001", 5000),
            ("T002", 70000)
        ]
    ),

    (
        "C102",
        "INACTIVE",
        [
            ("T003", 20000)
        ]
    ),

    (
        "C103",
        "ACTIVE",
        [
            ("T004", 120000),
            ("T005", 3000),
            ("T006", 45000)
        ]
    ),

    (
        "C104",
        "ACTIVE",
        []
    )
]

rdd = sc.parallelize(customers, 2)


REQUIREMENT:


STEP 1 — filter()

Keep only customers where:

status == "ACTIVE"



STEP 2 — flatMap()

Convert every transaction into an individual record.

Return:

(
    customer_id,
    transaction_id,
    amount
)


C104 has no transactions.

Therefore:

C104
    → 0 output records



STEP 3 — map()

Add transaction risk:

amount >= 50000
    → HIGH

amount >= 10000 and < 50000
    → MEDIUM

amount < 10000
    → LOW


Return:

(
    customer_id,
    transaction_id,
    amount,
    risk
)



STEP 4 — mapPartitionsWithIndex()

Create one summary record per partition.

Calculate:

(
    partition_id,
    transaction_count,
    total_amount,
    high_risk_count
)


EXPECTED TRANSACTION DATA BEFORE PARTITION SUMMARY:

[
    ("C101", "T001", 5000, "LOW"),

    ("C101", "T002", 70000, "HIGH"),

    ("C103", "T004", 120000, "HIGH"),

    ("C103", "T005", 3000, "LOW"),

    ("C103", "T006", 45000, "MEDIUM")
]


EXPECTED FINAL OUTPUT FORMAT:

[
    (
        0,
        transaction_count,
        total_amount,
        high_risk_count
    ),

    (
        1,
        transaction_count,
        total_amount,
        high_risk_count
    )
]


Exact partition summary values depend on how Spark
distributes the intermediate records.